# Policy Matrix Walkthrough

Phase 6 Increment 3 notebook for adaptive-vs-rule matrix analysis.

Workflow:
1. Optionally regenerate matrix artifacts from canonical preset.
2. Load latest matrix report.
3. Inspect rankings and baseline deltas.
4. Render a compact plot for top-ranked conditions.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys

def resolve_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not resolve repository root")

ROOT = resolve_repo_root(Path.cwd())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

ROOT


In [ ]:
RUN_GENERATORS = False

if RUN_GENERATORS:
    env = dict(os.environ)
    env["PYTHONPATH"] = "src"
    command = [
        sys.executable,
        "scripts/run_policy_experiment_matrix.py",
        "--config",
        "configs/experiments/policy_experiment_matrix_baseline.json",
    ]
    print("RUN", " ".join(command))
    subprocess.run(command, cwd=ROOT, env=env, check=True)
else:
    print("RUN_GENERATORS is False; using existing matrix artifacts")


In [ ]:
report_candidates = sorted(
    (ROOT / "artifacts" / "reports").glob("policy_experiment_matrix_*.json"),
    key=lambda path: path.stat().st_mtime,
)
if not report_candidates:
    raise FileNotFoundError("No policy matrix report found. Run scripts/run_policy_experiment_matrix.py")

report_path = report_candidates[-1]
report = json.loads(report_path.read_text(encoding="utf-8"))
report_path


In [ ]:
metadata = report["matrix_metadata"]
top_ranked = report["summary_rankings"]["lowest_final_compromised"][:5]
baseline_deltas = report["comparison_to_baseline"]

summary = {
    "scenario_id": metadata["scenario_id"],
    "seed_count": metadata["seed_count"],
    "horizon": metadata["horizon"],
    "condition_count": metadata["condition_count"],
    "include_ablations": metadata["include_ablations"],
}

summary, top_ranked[0], baseline_deltas[0]


In [ ]:
import matplotlib.pyplot as plt

labels = [entry["condition_id"] for entry in top_ranked]
values = [entry["final_compromised_mean"] for entry in top_ranked]

figure, axis = plt.subplots(figsize=(10, 4))
axis.bar(labels, values, color="#1d4ed8")
axis.set_title("Top 5 Conditions By Lowest Final Compromised Mean")
axis.set_ylabel("final_compromised_mean")
axis.tick_params(axis="x", labelrotation=35)
figure.tight_layout()
figure
